**Historical evaluation: Relay, not the revised Learn + team-files workload.**

This optional companion preserves the reproducible threshold-selection experiment from the earlier fictional Relay recipe. Its 0.96 threshold, labels, rubric, scores, and saved outputs apply only to that historical corpus. They do **not** establish the accuracy, operating threshold, or live runnability of the current [Learn + indexed team-files recipe](https://github.com/microsoft-foundry/forgebook/blob/main/notebooks/foundry-iq-decision-1-gate.ipynb).

Run from `notebooks`. The default path is offline and grades evidence sufficiency, not generated-answer correctness. Use the mechanics as a reference: capture new live evidence, label each retrieved bundle rather than guessing from the question, keep scenario families together, choose on development, and assess once on held-out cases. Do not carry the historical numbers over to a new workload.

## 1. Load historical evidence and scenario splits

The original Relay documents, questions, traces and single-call scores remain unchanged in `data/foundry-iq-decision-1-gate/`. The historical median-score and risk-tier experiments are not used here.

In [1]:
import hashlib
import json
import math
import sys
from pathlib import Path

DATA = Path("data/foundry-iq-decision-1-gate/")
sys.path.insert(0, str(DATA.resolve()))
from replay_retrieval import DOCUMENTS, QUESTIONS, recorded_retrieval

groups = json.loads((DATA / "evaluation-groups.json").read_text(encoding="utf-8"))
assignments = {}
for name, group in groups.items():
    for qid in group["questions"]:
        assert qid not in assignments, f"Question appears in multiple groups: {qid}"
        assignments[qid] = {"group": name, "split": group["split"]}
assert set(assignments) == set(QUESTIONS)
print({split: sum(row["split"] == split for row in assignments.values()) for split in ("dev", "test")})
print("q39 / q42:", assignments["q39"], assignments["q42"])

{'dev': 26, 'test': 34}
q39 / q42: {'group': 'networking_and_egress', 'split': 'test'} {'group': 'networking_and_egress', 'split': 'test'}


## 2. Inspect or regenerate the single-call scores

`RESCORE=False` reads the shipped results. To regenerate, install the main recipe's packages, sign in, set `DECISION_ENDPOINT` (and optionally `DECISION_DEPLOYMENT`), and change it to `True`. This makes 120 scoring requests: 60 questions at two recorded retrieval efforts. It does **not** rerun Foundry IQ.

Each request uses the historical Relay rubric stored in the metadata and the historical state shape, not the current Learn recipe's rubric. Requests are paced; a failed response raises rather than turning into a fabricated score. New files are written only after every request succeeds. The metadata binds scores to the questions, scenario split, evidence text, rubric, and recorded trace file.

In [2]:
RESCORE = False
SCORE_FILE = DATA / "single-call-scores.jsonl"
META_FILE = DATA / "single-call-scores.meta.json"


def digest(path):
    return hashlib.sha256(path.read_bytes().replace(b"\r\n", b"\n")).hexdigest()


def corpus_digest():
    return hashlib.sha256(json.dumps(DOCUMENTS, sort_keys=True).encode()).hexdigest()


if RESCORE:
    import os
    import time
    from datetime import datetime, timezone
    import httpx
    from azure.identity import DefaultAzureCredential
    from dotenv import load_dotenv

    load_dotenv(".env", override=True)
    rubric = {
        "type": "noul",
        "instructions": "Does the provided evidence contain enough information to fully answer the support question?",
        "criteria": {
            "true": ("The evidence explicitly covers every material part of the question, applies to "
                     "the relevant product version and plan, and leaves no unresolved contradictions."),
            "false": ("A required fact is missing, the version or plan does not match, the evidence "
                      "conflicts without a resolution, or answering would require assumed knowledge."),
        },
    }
    model = os.getenv("DECISION_DEPLOYMENT", "Decision-1")
    url = os.environ["DECISION_ENDPOINT"].rstrip("/") + "/providers/microsoft/v1/systemone"
    new_rows = []
    with DefaultAzureCredential() as credential, httpx.Client(timeout=60) as client:
        for qid, question in QUESTIONS.items():
            for effort in ("low", "medium"):
                evidence = recorded_retrieval(question["question"], effort)
                state = json.dumps({"question": question["question"], "evidence": evidence["docs"]})
                started = time.perf_counter()
                response = client.post(
                    url, headers={"Authorization": f"Bearer {credential.get_token('https://cognitiveservices.azure.com/.default').token}"},
                    json={"model": model, "state": state, "questions": {"sufficient": rubric}},
                )
                response.raise_for_status()
                payload = response.json()
                p = payload["answers"]["sufficient"]["noul"]
                assert not isinstance(p, bool) and isinstance(p, (int, float)) and 0 <= p <= 1
                new_rows.append({
                    "qid": qid, "effort": effort, "p": p,
                    "doc_ids": [doc["id"] for doc in evidence["docs"]],
                    "decision_ms": round((time.perf_counter() - started) * 1000),
                    "usage": payload["usage"],
                })
                time.sleep(1.5)
    metadata = {
        "scored_at": datetime.now(timezone.utc).isoformat(),
        "retrieval_collected_at": "2026-10-12",
        "retrieval_mode": "recorded", "scoring_mode": "live",
        "model": model, "scores_per_evidence_bundle": 1, "rubric": rubric,
        "questions_sha256": digest(DATA / "questions.jsonl"),
        "groups_sha256": digest(DATA / "evaluation-groups.json"),
        "traces_sha256": digest(DATA / "traces.jsonl"),
        "corpus_sha256": corpus_digest(),
    }
    SCORE_FILE.write_text("".join(json.dumps(row) + "\n" for row in new_rows), encoding="utf-8")
    META_FILE.write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")

metadata = json.loads(META_FILE.read_text(encoding="utf-8"))
assert metadata["scores_per_evidence_bundle"] == 1
assert metadata["questions_sha256"] == digest(DATA / "questions.jsonl")
assert metadata["groups_sha256"] == digest(DATA / "evaluation-groups.json")
assert metadata["traces_sha256"] == digest(DATA / "traces.jsonl")
assert metadata["corpus_sha256"] == corpus_digest()
scores = [json.loads(line) for line in SCORE_FILE.read_text(encoding="utf-8").splitlines()]
expected = {(qid, effort) for qid in QUESTIONS for effort in ("low", "medium")}
assert len(scores) == 120 and {(row["qid"], row["effort"]) for row in scores} == expected
for row in scores:
    assert not isinstance(row["p"], bool) and isinstance(row["p"], (int, float)) and 0 <= row["p"] <= 1
    evidence = recorded_retrieval(QUESTIONS[row["qid"]]["question"], row["effort"])
    assert row["doc_ids"] == [doc["id"] for doc in evidence["docs"]]
print(f"{len(scores)} single-call scores; live scoring date: {metadata['scored_at']}")
print("Retrieval: recorded, collected", metadata["retrieval_collected_at"])
print("Model tokens:", {key: sum(row["usage"].get(key, 0) for row in scores)
                        for key in ("input_tokens", "output_tokens")})

120 single-call scores; live scoring date: 2026-10-20T11:46:14.974492+00:00
Retrieval: recorded, collected 2026-10-12
Model tokens: {'input_tokens': 124756, 'output_tokens': 120}


## 3. Choose one threshold on development only

A trace is sufficient when its document IDs contain a complete labeled evidence set. These labels describe retrieved evidence, not generated-answer correctness. They depend on the synthetic corpus and must be replaced for another workload.

Round observed `low`-effort development scores up to two decimals, then select the lowest candidate that accepts at least one case and accepts no insufficient-evidence case on development. This is a deliberately simple **demo policy**, not a claim of zero production errors. Test scores never select the threshold. If no such threshold exists, the policy hands off everything.

In [3]:
def sufficient(row):
    question = QUESTIONS[row["qid"]]
    return any(set(required) <= set(row["doc_ids"]) for required in question["evidence_sets"])


def choose_threshold(rows):
    for threshold in sorted({math.ceil(row["p"] * 100) / 100 for row in rows}):
        accepted = [row for row in rows if row["p"] >= threshold]
        if accepted and all(sufficient(row) for row in accepted):
            return threshold
    return None


by_key = {(row["qid"], row["effort"]): row for row in scores}
dev = [row for row in scores if row["effort"] == "low" and assignments[row["qid"]]["split"] == "dev"]
threshold = choose_threshold(dev)
print("Development questions:", len(dev))
print("Selected threshold:", threshold if threshold is not None else "always hand off")
print("This is a historical Relay threshold, not the current Learn workload's operating point.")

Development questions: 26
Selected threshold: 0.96
This is a historical Relay threshold, not the current Learn workload's operating point.


## 4. Compare policies on the same held-out questions

**Acceptance** is answered questions / all held-out questions. **Answerable-question coverage** is accepted cases with sufficient evidence / questions answerable somewhere in the corpus. **Insufficient-evidence acceptance** is accepted cases without sufficient evidence / accepted cases.

The baseline accepts every question after `low` retrieval. The basic gate uses the same `low` evidence. The optional retry policy examines `medium` only when `low` fails the threshold, and uses the same threshold. Every row counts each question once; it never pools four efforts as independent samples.

In [4]:
def evaluate(qids, threshold, retry=False, always_answer=False):
    outcomes = []
    for qid in qids:
        last = by_key[(qid, "low")]
        accepted = always_answer or (threshold is not None and last["p"] >= threshold)
        attempts = 1
        if retry and not accepted:
            last = by_key[(qid, "medium")]
            attempts = 2
            accepted = threshold is not None and last["p"] >= threshold
        outcomes.append({"qid": qid, "accepted": accepted, "sufficient": sufficient(last),
                         "answerable": QUESTIONS[qid]["answerable"], "attempts": attempts})
    return outcomes


test_qids = sorted(qid for qid, row in assignments.items() if row["split"] == "test")
policies = {
    "Always answer after low": evaluate(test_qids, threshold, always_answer=True),
    "Single-call gate, low": evaluate(test_qids, threshold),
    "Gate + one medium retry": evaluate(test_qids, threshold, retry=True),
}
for name, outcomes in policies.items():
    accepted = [row for row in outcomes if row["accepted"]]
    bad = sum(not row["sufficient"] for row in accepted)
    supported = len(accepted) - bad
    answerable = sum(row["answerable"] for row in outcomes)
    print(name)
    print(f"  Accepted: {len(accepted)}/{len(outcomes)} ({len(accepted)/len(outcomes):.1%})")
    print(f"  Insufficient evidence among accepted: {bad}/{len(accepted)}")
    print(f"  Answerable-question coverage: {supported}/{answerable} ({supported/answerable:.1%})")
    print(f"  Retrieval/scoring attempts: {sum(row['attempts'] for row in outcomes)}")

Always answer after low
  Accepted: 34/34 (100.0%)
  Insufficient evidence among accepted: 17/34
  Answerable-question coverage: 17/17 (100.0%)
  Retrieval/scoring attempts: 34
Single-call gate, low
  Accepted: 14/34 (41.2%)
  Insufficient evidence among accepted: 0/14
  Answerable-question coverage: 14/17 (82.4%)
  Retrieval/scoring attempts: 34
Gate + one medium retry
  Accepted: 15/34 (44.1%)
  Insufficient evidence among accepted: 0/15
  Answerable-question coverage: 15/17 (88.2%)
  Retrieval/scoring attempts: 54


Small counts matter. Even zero observed insufficient-evidence acceptances would not establish zero risk; an application can also generate an unfaithful reply from sufficient evidence. This notebook does not measure that second failure mode.

The split avoids related-question leakage by grouping scenarios, but it leaves few scenario families in each partition. All documents and questions are synthetic and were developed together. Treat these results as a reproducible integration check, not an independent or representative support benchmark.

## 5. Inspect the cases the policy got wrong

Inspect false acceptances and handoffs on answerable questions before adding complexity. A stricter threshold cannot recover missing documents. A higher retrieval effort can change which documents are returned, not guarantee better evidence.

In [5]:
for name, outcomes in policies.items():
    if name == "Always answer after low":
        continue
    print("\n" + name)
    for row in outcomes:
        if (row["accepted"] and not row["sufficient"]) or (not row["accepted"] and row["answerable"]):
            print(row["qid"], "accepted insufficient evidence" if row["accepted"] else "handed off answerable question",
                  "-", QUESTIONS[row["qid"]]["question"])


Single-call gate, low
q15 handed off answerable question - After how many consecutive failures does Relay auto-disable an endpoint today?
q26 handed off answerable question - We are on Enterprise in ap-south running v3.4. Can we use mutual TLS, and what should our receiver keep relying on to check that deliveries come from Relay?
q30 handed off answerable question - If my Pro endpoint is auto-disabled, how long are new events held, and how long do failed deliveries stay in the DLQ?

Gate + one medium retry
q26 handed off answerable question - We are on Enterprise in ap-south running v3.4. Can we use mutual TLS, and what should our receiver keep relying on to check that deliveries come from Relay?
q30 handed off answerable question - If my Pro endpoint is auto-disabled, how long are new events held, and how long do failed deliveries stay in the DLQ?


## Historical experiments

The original `traces.jsonl` also contains `minimal` and `auto`, repeated probabilities in `p`, alternate-rubric scores in `p_alt`, reranker scores, and planner token counts. Its provenance is in `traces.meta.json`. Those are historical measurements under the **older rubric**, not evidence for the new single-call threshold.

The old `policy.json` records a different three-call, risk-tier cascade and must not be loaded by the introductory recipe. Its 18/40 accepted questions meant 45% overall acceptance; 18/21 meant 86% answerable-question coverage. Neither described evaluated generated-answer accuracy.

Before extending the recipe: collect fresh retrieval on your own support docs, label complete evidence sets, split related incidents together, choose a threshold on development, and report held-out counts. Evaluate generated answers separately. Add repeated scoring, more retrieval attempts, or multiple policies only when the measured benefit justifies their requests, latency, and maintenance.